# OpenMM CHARMM/PME GPCRmd MD Preview

This notebook previews a short complete OpenMM MD run from the local GPCRmd CHARMM inputs: PSF, PDB, PRM, PME electrostatics, HBond constraints, rigid water, and Langevin dynamics. The default artifact is 200 OpenMM `OpenCL` steps sampled every 2 steps, giving 101 saved frames.

This is a short stability/visualization trial. Over 0.05 ps, ligand translation is expected to be small; large ligand migration requires much longer physical simulation or a guided protocol.

In [ ]:
import json
import os
import sys
from dataclasses import replace
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "helpers").exists():
    NOTEBOOK_DIR = Path("notebooks/ligand-receptor-motion").resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from helpers.motion_analysis import (  # noqa: E402
    ProcessedTrajectory,
    analysis_tables,
    ligand_com_displacement,
    trajectory_quality_report,
)
from helpers.visualization import make_ligand_motion_figure, playback_table  # noqa: E402

DATASET_NAME = os.environ.get("OPENMM_MD_DATASET", "729-200-opencl-charmm-pme-dense-preview")
DATA_DIR = NOTEBOOK_DIR / "data/openmm-md" / DATASET_NAME
TRAJECTORY_PATH = DATA_DIR / "processed_trajectory.npz"
REPORT_PATH = DATA_DIR / "openmm_charmm_md_run_report.json"
VIEW_STRIDE = int(os.environ.get("OPENMM_MD_VIEW_STRIDE", "1"))

## Artifact Status

Regenerate the default full-MD preview outside the notebook with:

```bash
uv run python scripts/run_openmm_gpcrmd_charmm_md.py \
  --out notebooks/ligand-receptor-motion/data/openmm-md/729-200-opencl-charmm-pme-dense-preview \
  --steps 200 --sample-interval 2 --platform OpenCL \
  --dt-ps 0.00025 --temperature 310 --friction 0.1 \
  --minimize-steps 500 --positions-source prepared
```

In [ ]:
if not TRAJECTORY_PATH.exists() or not REPORT_PATH.exists():
    missing = [str(path) for path in (TRAJECTORY_PATH, REPORT_PATH) if not path.exists()]
    raise FileNotFoundError(f"Missing OpenMM MD artifact: {missing}")

trajectory = ProcessedTrajectory.load(TRAJECTORY_PATH)
report = json.loads(REPORT_PATH.read_text())
source = trajectory.source
ligand_displacement = ligand_com_displacement(trajectory)

display(
    pd.DataFrame(
        [
            {
                "dataset": DATASET_NAME,
                "status": report.get("status"),
                "engine": report.get("engine"),
                "workflow": report.get("workflow"),
                "platform": report.get("platform"),
                "device": report.get("platform_properties", {}).get("DeviceName"),
                "steps": report.get("steps"),
                "sample_interval": report.get("sample_interval"),
                "saved_frames": report.get("sampled_frame_count"),
                "simulated_time_ps": report.get("simulated_time_ps"),
                "dt_ps": report.get("dt_ps"),
                "steps_per_second": report.get("integration_steps_per_second"),
                "cutoff_A": report.get("cutoff_A"),
                "switch_A": report.get("switch_A"),
                "minimize_steps": report.get("minimize_steps"),
                "ligand_final_displacement_A": float(ligand_displacement[-1]),
                "ligand_max_displacement_A": float(ligand_displacement.max()),
            }
        ]
    )
)

display(Markdown(f"**Preview note:** {source.get('note', '')}"))
display(playback_table(trajectory))

quality_report = trajectory_quality_report(trajectory, max_constraint_error_A=0.0)
display(Markdown("**Visualization Quality Report**"))
display(pd.DataFrame([{k: v for k, v in quality_report.items() if k != "warnings"}]))
for warning in quality_report["warnings"]:
    display(Markdown(f"- {warning}"))

## Trajectory Viewer

The viewer shows saved full-MD frames in a PBC-corrected display frame. This is unguided MD, so the ligand mostly fluctuates in place over the 5 ps default trial.

In [ ]:
if VIEW_STRIDE > 1:
    viewer_trajectory = replace(
        trajectory,
        positions=trajectory.positions[::VIEW_STRIDE],
        time_ps=trajectory.time_ps[::VIEW_STRIDE],
        source={**trajectory.source, "viewer_stride": VIEW_STRIDE},
    )
else:
    viewer_trajectory = trajectory

figure = make_ligand_motion_figure(
    viewer_trajectory,
    title=(
        "OpenMM CHARMM/PME GPCRmd full-MD preview "
        f"({viewer_trajectory.frame_count} displayed frames)"
    ),
    frame_interval_ms=60,
)
figure.show()

## Diagnostics

In [ ]:
diagnostics = pd.DataFrame(
    {
        "step": report["sample_interval"] * pd.Series(range(report["sampled_frame_count"])),
        "time_ps": trajectory.time_ps,
        "potential_energy_kj_mol": report["potential_energy_kj_mol"],
        "kinetic_energy_kj_mol": report["kinetic_energy_kj_mol"],
        "temperature_K": report["temperature_K"],
        "ligand_com_displacement_A": ligand_displacement,
    }
)
display(diagnostics.tail())

energy_fig = px.line(
    diagnostics,
    x="time_ps",
    y=["potential_energy_kj_mol", "kinetic_energy_kj_mol"],
    title="OpenMM CHARMM/PME energy diagnostics",
    labels={"value": "kJ/mol", "variable": "term"},
)
energy_fig.show()

temp_fig = px.line(
    diagnostics,
    x="time_ps",
    y="temperature_K",
    title="OpenMM CHARMM/PME temperature",
)
temp_fig.show()

motion_fig = px.line(
    diagnostics,
    x="time_ps",
    y="ligand_com_displacement_A",
    title="Ligand COM displacement in full OpenMM MD frames",
)
motion_fig.show()

## Interaction Tables

In [ ]:
tables = analysis_tables(trajectory)
for name, table in tables.items():
    display(Markdown(f"**{name.replace('_', ' ').title()}**"))
    display(table)